In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.dim_seller (
    seller_key BIGINT GENERATED ALWAYS AS IDENTITY,
    seller_id STRING NOT NULL,
    seller_zip_code_prefix STRING,
    seller_city STRING,
    seller_state STRING,
    effective_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN,
    gold_updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Gold seller dimension - SCD Type 2. Tracks history of seller location changes. Grain: one row per seller_id per version.';

In [0]:
display(spark.sql("SELECT * FROM ecommerce_dev.silver.sellers LIMIT 5"))

seller_id,seller_zip_code_prefix,seller_city,seller_state,bronze_ingested_at
3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP,2026-08-04T00:56:03.525Z
d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP,2026-08-04T00:56:03.525Z
ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ,2026-08-04T00:56:03.525Z
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,Sao Paulo,SP,2026-08-04T00:56:03.525Z
51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP,2026-08-04T00:56:03.525Z


### Step 1: Stage the source

In [0]:
from pyspark.sql.functions import *

silver_sellers = spark.table("ecommerce_dev.silver.sellers")
silver_sellers.createOrReplaceTempView("stg_sellers")

### Step 2: Close out changed current rows

In [0]:
spark.sql("""
    MERGE INTO ecommerce_dev.gold.dim_seller AS target
    USING stg_sellers AS source
    ON target.seller_id = source.seller_id AND target.is_current = true

    WHEN MATCHED AND (
        target.seller_zip_code_prefix <> source.seller_zip_code_prefix OR
        target.seller_city <> source.seller_city OR
        target.seller_state <> source.seller_state
    ) THEN UPDATE SET
        target.end_date = current_timestamp(),
        target.is_current = false,
        target.gold_updated_at = current_timestamp()
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Step 3: Insert new sellers + new versions of changed sellers

In [0]:
is_initial_load = spark.sql(
    "SELECT count(*) as ct FROM ecommerce_dev.gold.dim_seller"
).collect()[0]['ct'] == 0

effective_date_expr = "TIMESTAMP('1900-01-01')" if is_initial_load else "current_timestamp()"

spark.sql(f"""
    MERGE INTO ecommerce_dev.gold.dim_seller AS target
    USING stg_sellers AS source
    ON target.seller_id = source.seller_id AND target.is_current = true

    WHEN NOT MATCHED THEN INSERT (
        seller_id, seller_zip_code_prefix, seller_city, seller_state,
        effective_date, end_date, is_current, gold_updated_at
    ) VALUES (
        source.seller_id, source.seller_zip_code_prefix, source.seller_city, source.seller_state,
        {effective_date_expr}, NULL, true, current_timestamp()
    )
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Step 4: NOT NULL + PK, comment, verify

In [0]:
%sql
ALTER TABLE ecommerce_dev.gold.dim_seller ALTER COLUMN seller_key SET NOT NULL;

ALTER TABLE ecommerce_dev.gold.dim_seller 
ADD CONSTRAINT pk_dim_seller PRIMARY KEY (seller_key);

COMMENT ON TABLE ecommerce_dev.gold.dim_seller IS 
'Gold seller dimension - SCD Type 2. Tracks history of seller_zip_code_prefix, seller_city, seller_state changes. Grain: one row per seller_id per version. Surrogate key: seller_key. is_current flags the active version.';

In [0]:
# Sanity checks: current-row count should match Silver seller count exactly (first run = no history yet)
silver_ct = spark.table("ecommerce_dev.silver.sellers").count()
gold_current_ct = spark.sql("SELECT count(*) as ct FROM ecommerce_dev.gold.dim_seller WHERE is_current = true").collect()[0]['ct']
gold_total_ct = spark.table("ecommerce_dev.gold.dim_seller").count()

print(f"Silver: {silver_ct} | Gold current: {gold_current_ct} | Gold total: {gold_total_ct}")
print(f"Current matches Silver: {silver_ct == gold_current_ct}")
print(f"No history yet (total == current) on first run: {gold_total_ct == gold_current_ct}")

display(spark.sql("SELECT * FROM ecommerce_dev.gold.dim_seller LIMIT 5"))

Silver: 3095 | Gold current: 3095 | Gold total: 3095
Current matches Silver: True
No history yet (total == current) on first run: True


seller_key,seller_id,seller_zip_code_prefix,seller_city,seller_state,effective_date,end_date,is_current,gold_updated_at
1,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP,2026-08-09T17:38:03.122Z,null,true,2026-08-09T17:38:03.122Z
2,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP,2026-08-09T17:38:03.122Z,null,true,2026-08-09T17:38:03.122Z
3,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ,2026-08-09T17:38:03.122Z,null,true,2026-08-09T17:38:03.122Z
4,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,Sao Paulo,SP,2026-08-09T17:38:03.122Z,null,true,2026-08-09T17:38:03.122Z
5,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP,2026-08-09T17:38:03.122Z,null,true,2026-08-09T17:38:03.122Z
